<a href="https://colab.research.google.com/github/ewertonlna-cloud/DATA_GUARDA777/blob/main/DATA_PY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

produtos = ["Notebook", "Smartphone", "Tablet", "Monitor", "Teclado", "Mouse", "Headphone", "Webcam"]
regioes = ["Norte", "Sul", "Sudeste", "Nordeste", "Centro-Oeste"]
vendedores = ["Ana", "Bruno", "Carlos", "Diana", "Eduardo"]

n = 500
datas = [date(2024, 1, 1) + timedelta(days=random.randint(0, 364)) for _ in range(n)]

df = pd.DataFrame({
    "data":       datas,
    "produto":    random.choices(produtos, k=n),
    "regiao":     random.choices(regioes, k=n),
    "vendedor":   random.choices(vendedores, k=n),
    "quantidade": np.random.randint(1, 20, n),
    "preco_unit": np.random.choice([799, 1999, 1499, 999, 199, 149, 299, 249], n),
})

df["total"] = df["quantidade"] * df["preco_unit"]
df["mes"] = pd.to_datetime(df["data"]).dt.month

df.to_csv("vendas.csv", index=False)
print("✅ vendas.csv gerado com sucesso!")

✅ vendas.csv gerado com sucesso!


In [2]:
import pandas as pd
import sqlite3

def get_connection(csv_path="vendas.csv"):
    df = pd.read_csv(csv_path)
    conn = sqlite3.connect(":memory:")
    df.to_sql("vendas", conn, index=False, if_exists="replace")
    return conn

def query(sql, csv_path="vendas.csv"):
    conn = get_connection(csv_path)
    return pd.read_sql_query(sql, conn)

# --- Queries prontas ---

def total_por_produto():
    return query("""
        SELECT produto,
               SUM(quantidade) AS unidades_vendidas,
               SUM(total)      AS receita_total
        FROM vendas
        GROUP BY produto
        ORDER BY receita_total DESC
    """)

def total_por_regiao():
    return query("""
        SELECT regiao,
               SUM(total) AS receita_total
        FROM vendas
        GROUP BY regiao
        ORDER BY receita_total DESC
    """)

def vendas_por_mes():
    return query("""
        SELECT mes,
               COUNT(*)   AS num_vendas,
               SUM(total) AS receita_total
        FROM vendas
        GROUP BY mes
        ORDER BY mes
    """)

def top_vendedores():
    return query("""
        SELECT vendedor,
               COUNT(*)   AS num_vendas,
               SUM(total) AS receita_total
        FROM vendas
        GROUP BY vendedor
        ORDER BY receita_total DESC
    """)